# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 个人GitHub项目说明

- 每名学生独立完成本Notebook；
- 输入文件固定为`data/E Commerce Dataset.xlsx`；
- 输出固定写入`output/day04_project/`；
- 不要提交教师演示Notebook或教师参考答案；
- 完成后重启内核并从头运行，再推送到个人GitHub仓库。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [6]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


def find_project_root(start=None):
    """从当前目录向上寻找种子项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "E Commerce Dataset.xlsx").exists():
            return candidate
    raise FileNotFoundError("未找到data/E Commerce Dataset.xlsx")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "E Commerce Dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：d:\ZZZshixi\muc-commerce-2-24012408\data\E Commerce Dataset.xlsx
项目输出目录：d:\ZZZshixi\muc-commerce-2-24012408\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [7]:
# 在此写下你的答案：
# 1.电商客户流失数据
# 2.Churm
# 3.CustomerID只是标识符，不是连续的数值

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [8]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    # 返回一个 DataFrame，至少包含：
    # 数据类型、缺失数量、缺失比例(%)、唯一值数量
    report = pd.DataFrame({
        '字段类型': data.dtypes,
        '缺失数量': data.isnull().sum(),
        '缺失比例(%)': (data.isnull().sum() / len(data) * 100).round(2),
        '唯一值数量': data.nunique()
    })
    report.index.name = '字段'
    return report


# 生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)

,字段类型,缺失数量,缺失比例(%),唯一值数量
字段,,,,
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,object,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,object,0,0.00,7
Gender,object,0,0.00,2
HourSpendOnApp,float64,255,4.53,6


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [9]:
# 完成项目初始审计
print("完全重复行数：", raw_df.duplicated().sum())
print("CustomerID 重复数量：", raw_df["CustomerID"].duplicated().sum())
print(raw_df["Churn"].value_counts())
print("流失率：", raw_df["Churn"].mean() * 100, "%")

for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())

完全重复行数： 0
CustomerID 重复数量： 0
Churn
0    4682
1     948
Name: count, dtype: int64
流失率： 16.838365896980463 %

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [10]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [11]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """  
    # 复制数据，避免覆盖原始数据   
    df = data.copy()
    # 创建日志列表 logs
    logs = []

    # 记录初始状态
    initial_records = len(df)

    # 删除完全重复行
    before_dedup = len(df)
    df = df.drop_duplicates()
    after_dedup = len(df)
    duplicated_count = before_dedup - after_dedup
    logs.append({
        "step": "删除完全重复行",
        "rule": "drop_duplicates() 删除所有列完全相同的重复行",
        "records_before": before_dedup,
        "records_after": after_dedup,
        "affected_records": duplicated_count
    })

    # 对 NUMERIC_MISSING_COLS 使用中位数填补
    for col in NUMERIC_MISSING_COLS:
        if col in df.columns:
            missing_count = df[col].isna().sum()
            if missing_count > 0:
                median_val = df[col].median()
                before_fill = len(df)
                df[col] = df[col].fillna(median_val)
                after_fill = len(df)
                logs.append({
                    "step": f"缺失值填补: {col}",
                    "rule": f"使用中位数 {median_val:.2f} 填补缺失值",
                    "records_before": before_fill,
                    "records_after": after_fill,
                    "affected_records": missing_count
                })

    # 对 CATEGORY_MAPPINGS 完成类别标准化
    for col, mapping in CATEGORY_MAPPINGS.items():
        if col in df.columns:
            for old_val, new_val in mapping.items():
                affected = (df[col] == old_val).sum()
                if affected > 0:
                    before_map = len(df)
                    df[col] = df[col].replace({old_val: new_val})
                    after_map = len(df)
                    logs.append({
                        "step": f"类别标准化: {col}",
                        "rule": f"'{old_val}' -> '{new_val}'",
                        "records_before": before_map,
                        "records_after": after_map,
                        "affected_records": int(affected)
                    })

    # 将 Churn 和 Complain 转为整数类型
    int_cols = ["Churn", "Complain"]
    for col in int_cols:
        if col in df.columns:
            before_cast = len(df)
            df[col] = pd.to_numeric(df[col], errors="coerce")
            # 填补转换后可能产生的 NaN（如果有）
            nan_after_cast = df[col].isna().sum()
            if nan_after_cast > 0:
                df[col] = df[col].fillna(0)
            df[col] = df[col].astype(int)
            after_cast = len(df)
            logs.append({
                "step": f"数据类型转换: {col}",
                "rule": "转换为 int 类型，无法解析的值转为 0",
                "records_before": before_cast,
                "records_after": after_cast,
                "affected_records": 0 if nan_after_cast == 0 else int(nan_after_cast)
            })

    # 构建日志 DataFrame
    cleaning_log = pd.DataFrame(logs)

    # 添加汇总行
    if not cleaning_log.empty:
        summary = pd.DataFrame([{
            "step": "汇总",
            "rule": "全部清洗步骤完成",
            "records_before": initial_records,
            "records_after": len(df),
            "affected_records": initial_records - len(df)
        }])
        cleaning_log = pd.concat([cleaning_log, summary], ignore_index=True)

    cleaned_df = df.reset_index(drop=True)
    return cleaned_df, cleaning_log

### 任务 3：运行清洗函数并查看日志

In [12]:
# 执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

display(cleaning_log)
cleaned_df.head()

,step,rule,records_before,records_after,affected_records
0,删除完全重复行,drop_duplicates() 删除所有列完全相同的重复行,5630,5630,0
1,缺失值填补: Tenure,使用中位数 9.00 填补缺失值,5630,5630,264
2,缺失值填补: WarehouseToHome,使用中位数 14.00 填补缺失值,5630,5630,251
3,缺失值填补: HourSpendOnApp,使用中位数 3.00 填补缺失值,5630,5630,255
4,缺失值填补: OrderAmountHikeFromlastYear,使用中位数 15.00 填补缺失值,5630,5630,265
5,缺失值填补: CouponUsed,使用中位数 1.00 填补缺失值,5630,5630,256
6,缺失值填补: OrderCount,使用中位数 2.00 填补缺失值,5630,5630,258
7,缺失值填补: DaySinceLastOrder,使用中位数 3.00 填补缺失值,5630,5630,307
8,类别标准化: PreferredLoginDevice,'Phone' -> 'Mobile Phone',5630,5630,1231
9,类别标准化: PreferredPaymentMode,'COD' -> 'Cash on Delivery',5630,5630,365


字段,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [13]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# 构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
tenure_bins = [-1, 0, 6, 12, 24, 1000]
tenure_labels = ["新用户", "1-6个月", "7-12个月", "13-24个月", "两年以上老用户"]
cleaned_df["TenureGroup"] = pd.cut(cleaned_df["Tenure"], bins=tenure_bins, labels=tenure_labels, right=True)

# 新建 IsMobileLogin，移动端为 1，其他设备为 0
cleaned_df["IsMobileLogin"] = (cleaned_df["PreferredLoginDevice"] == "Mobile Phone").astype(int)

# 生成 outlier_report（每行对应一个待检查字段）
outlier_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_report = pd.DataFrame([
    {"字段": col, **iqr_outlier_summary(cleaned_df[col])}
    for col in outlier_cols if col in cleaned_df.columns
])

### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [14]:
# 完成业务规则检查
business_rule_report = pd.DataFrame({
    "规则": [
        "使用时长小于 0",
        "仓库距离小于 0",
        "订单数小于或等于 0",
        "返现金额小于 0"],
    "不合规记录数": [
        (cleaned_df["Tenure"] < 0).sum(),
        (cleaned_df["WarehouseToHome"] < 0).sum(),
        (cleaned_df["OrderCount"] <= 0).sum(),
        (cleaned_df["CashbackAmount"] < 0).sum()]
})
display(business_rule_report)

# 处理结论：未发现不合规记录

,规则,不合规记录数
0,使用时长小于 0,0
1,仓库距离小于 0,0
2,订单数小于或等于 0,0
3,返现金额小于 0,0


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [15]:
# 完成最终验收
quality_after = build_quality_report(cleaned_df)

assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique()
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique()
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique()
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns)

# 导出下列文件，使用 utf-8-sig 编码：
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")

# 输出 outlier_report 和 business_rule_report
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")

# 输出交付文件的路径
print(f"交付文件已保存至：{OUTPUT_DIR}")

交付文件已保存至：d:\ZZZshixi\muc-commerce-2-24012408\output\day04_project


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

## GitHub提交检查

- [ ] Notebook已重启内核并从头运行成功；
- [ ] `output/day04_project/`包含清洗后数据、质量报告、清洗日志和异常/业务规则报告；
- [ ] 原始Excel没有被覆盖；
- [ ] 清洗函数、处理日志和项目复盘均已完成；
- [ ] 已提交并推送到个人GitHub仓库。